In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

from pathlib import Path
WORKDIR = str(Path.cwd().parent) + '/'  # raiz do repositório
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/logs/"+data_hora

In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [3]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [4]:
## get dataset

# select only sensor data from smart watches
csv_gyro_dir = WORKDIR +"Data/Raw/phone/gyro"
csv_accel_dir = WORKDIR +"Data/Raw/phone/accel"

In [5]:
df_csv_gyro = combine_csv_data(csv_gyro_dir)

In [6]:
df_csv_gyro.columns = ['pessoa_gyro','atividade_gyro','tempo_gyro','giroscópio_x','giroscópio_Y','giroscópio_Z']

In [7]:
df_csv_gyro.head()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,252207918580802,-0.853210,0.297226,0.8901825;
1,1600,A,252207968934806,-0.875137,0.015472,0.16223145;
2,1600,A,252208019288809,-0.720169,0.388489,-0.28401184;
3,1600,A,252208069642813,-0.571640,1.227402,-0.2416687;
4,1600,A,252208119996817,-0.380493,1.202835,-0.21313477;


In [8]:
df_csv_gyro.shape

(3608635, 6)

In [9]:
df_csv_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [10]:
df_csv_gyro.head()

giroscópio_x  giroscópio_Y  \
pessoa_gyro atividade_gyro tempo_gyro                                    
1600        A              252207918580802     -0.853210      0.297226   
                           252207968934806     -0.875137      0.015472   
                           252208019288809     -0.720169      0.388489   
                           252208069642813     -0.571640      1.227402   
                           252208119996817     -0.380493      1.202835   

                                            giroscópio_Z  
pessoa_gyro atividade_gyro tempo_gyro                     
1600        A              252207918580802    0.8901825;  
                           252207968934806   0.16223145;  
                           252208019288809  -0.28401184;  
                           252208069642813   -0.2416687;  
                           252208119996817  -0.21313477;

In [11]:
df_csv_gyro.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3608635 entries, (1600, 'A', 252207918580802) to (1650, 'S', 357734138868000)
Data columns (total 3 columns):
 #   Column        Dtype  
---  ------        -----  
 0   giroscópio_x  float64
 1   giroscópio_Y  float64
 2   giroscópio_Z  object 
dtypes: float64(2), object(1)
memory usage: 259.3+ MB


In [12]:
df_csv_accel = combine_csv_data(csv_accel_dir)

In [13]:
df_csv_accel.shape

(4804403, 6)

In [14]:
df_csv_accel.columns = ['pessoa_acelelômetro','atividade_acelerômetro','tempo_acelelômetro','acelelômetro_x','acelelômetro_Y','acelelômetro_Z']

In [15]:
df_csv_accel.head()

,pessoa_acelelômetro,atividade_acelerômetro,tempo_acelelômetro,acelelômetro_x,acelelômetro_Y,acelelômetro_Z
0,1600,A,252207666810782,-0.364761,8.793503,1.0550842;
1,1600,A,252207717164786,-0.879730,9.768784,1.0169983;
2,1600,A,252207767518790,2.001495,11.109070,2.619156;
3,1600,A,252207817872794,0.450623,12.651642,0.18455505;
4,1600,A,252207868226798,-2.164352,13.928436,-4.4224854;


In [16]:
df_csv_accel.set_index(["pessoa_acelelômetro","atividade_acelerômetro","tempo_acelelômetro"],inplace=True)

In [17]:
df_csv_accel.head()

acelelômetro_x  \
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                   
1600                A                      252207666810782          -0.364761   
                                           252207717164786          -0.879730   
                                           252207767518790           2.001495   
                                           252207817872794           0.450623   
                                           252207868226798          -2.164352   

                                                               acelelômetro_Y  \
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                   
1600                A                      252207666810782           8.793503   
                                           252207717164786           9.768784   
                                           252207767518790          11.109070   
                                           252207817872794          12.651642   
                                           252207868226798          13.928436   

                                                              acelelômetro_Z  
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                 
1600                A                      252207666810782        1.0550842;  
                                           252207717164786        1.0169983;  
                                           252207767518790         2.619156;  
                                           252207817872794       0.18455505;  
                                           252207868226798       -4.4224854;

In [18]:
# Select rows with matching indices (inner join)
common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)
df1_filtered = df_csv_accel.loc[common_indices]
df2_filtered = df_csv_gyro.loc[common_indices]

# Concatenate horizontally
result = pd.concat([df1_filtered, df2_filtered], axis=1)

In [19]:
result.shape

(2787819, 6)

In [20]:
result.tail()

acelelômetro_x  acelelômetro_Y acelelômetro_Z  \
1650 S 357733977635000       -8.509504       -3.520679    -1.8226099;   
       357734017925000       -8.423284       -4.061953    -1.7603394;   
       357734058325000       -8.327483       -4.196074    -1.5711329;   
       357734098619000       -8.080796       -4.418811    -1.4154567;   
       357734138868000       -8.040081       -4.366121    -1.4968873;   

                        giroscópio_x  giroscópio_Y giroscópio_Z  
1650 S 357733977635000      0.438889      0.171508  0.16937704;  
       357734017925000      0.444215      0.158724  0.19707392;  
       357734058325000      0.519849      0.037284  0.19600865;  
       357734098619000      0.545415     -0.086286  0.17896444;  
       357734138868000      0.471912     -0.201335  0.23222765;

In [21]:
result.reset_index(inplace=True)

In [22]:
result.tail()

,level_0,level_1,level_2,acelelômetro_x,acelelômetro_Y,acelelômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
2787814,1650,S,357733977635000,-8.509504,-3.520679,-1.8226099;,0.438889,0.171508,0.16937704;
2787815,1650,S,357734017925000,-8.423284,-4.061953,-1.7603394;,0.444215,0.158724,0.19707392;
2787816,1650,S,357734058325000,-8.327483,-4.196074,-1.5711329;,0.519849,0.037284,0.19600865;
2787817,1650,S,357734098619000,-8.080796,-4.418811,-1.4154567;,0.545415,-0.086286,0.17896444;
2787818,1650,S,357734138868000,-8.040081,-4.366121,-1.4968873;,0.471912,-0.201335,0.23222765;


In [23]:
result.columns = ['pessoa_x','atividade','tempo','acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'] 

In [24]:
result.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,252207918580802,-4.332779,13.361191,-0.7188721;,-0.853210,0.297226,0.8901825;
1,1600,A,252207968934806,-0.319443,13.318359,-0.23202515;,-0.875137,0.015472,0.16223145;
2,1600,A,252208019288809,1.566452,9.515274,-0.01777649;,-0.720169,0.388489,-0.28401184;
3,1600,A,252208069642813,-0.323746,5.262665,0.32234192;,-0.571640,1.227402,-0.2416687;
4,1600,A,252208119996817,-1.811676,3.710510,1.3739319;,-0.380493,1.202835,-0.21313477;


In [25]:
result.atividade.value_counts().sort_index()

atividade
A    144976
B    146880
C    153292
D    173177
E    157932
F    159508
G    161633
H    157037
I    153099
J    163470
K    149138
L    157497
M    143515
O    139537
P    143199
Q    168502
R    158166
S    157261
Name: count, dtype: int64

In [26]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [27]:
result["atividade"] = result["atividade"].replace(to_replace=mapping_dict)

In [28]:
result.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,252207918580802,-4.332779,13.361191,-0.7188721;,-0.853210,0.297226,0.8901825;
1,1600,0,252207968934806,-0.319443,13.318359,-0.23202515;,-0.875137,0.015472,0.16223145;
2,1600,0,252208019288809,1.566452,9.515274,-0.01777649;,-0.720169,0.388489,-0.28401184;
3,1600,0,252208069642813,-0.323746,5.262665,0.32234192;,-0.571640,1.227402,-0.2416687;
4,1600,0,252208119996817,-1.811676,3.710510,1.3739319;,-0.380493,1.202835,-0.21313477;


In [29]:
result.tail()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
2787814,1650,17,357733977635000,-8.509504,-3.520679,-1.8226099;,0.438889,0.171508,0.16937704;
2787815,1650,17,357734017925000,-8.423284,-4.061953,-1.7603394;,0.444215,0.158724,0.19707392;
2787816,1650,17,357734058325000,-8.327483,-4.196074,-1.5711329;,0.519849,0.037284,0.19600865;
2787817,1650,17,357734098619000,-8.080796,-4.418811,-1.4154567;,0.545415,-0.086286,0.17896444;
2787818,1650,17,357734138868000,-8.040081,-4.366121,-1.4968873;,0.471912,-0.201335,0.23222765;


In [30]:
cols_to_check = ['acelerômetro_Z','giroscópio_Z']

In [31]:
result[cols_to_check] = result[cols_to_check].replace({';':''}, regex=True)

In [32]:
result.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,252207918580802,-4.332779,13.361191,-0.7188721,-0.853210,0.297226,0.8901825
1,1600,0,252207968934806,-0.319443,13.318359,-0.23202515,-0.875137,0.015472,0.16223145
2,1600,0,252208019288809,1.566452,9.515274,-0.01777649,-0.720169,0.388489,-0.28401184
3,1600,0,252208069642813,-0.323746,5.262665,0.32234192,-0.571640,1.227402,-0.2416687
4,1600,0,252208119996817,-1.811676,3.710510,1.3739319,-0.380493,1.202835,-0.21313477


In [33]:
result.to_csv(WORKDIR + 'Data/processed/activity_recognition_data_phone_join_multi_index.csv', index=False)